# Discounting and SVCJ identification robustness

This notebook is diagnostic. It does **not** tune `lambda_TS` and does not alter the baseline calibration. For the current remediation pass the fixed specifications are ordinary WLS (`lambda_TS = 0`) and the provisional penalized WLS baseline (`lambda_TS = 100`). Deterministic discount rates 0%, 2.5%, 5%, and 7.5% are applied to already-computed regular USD futures-option values before conversion to coin units.

Promotion rule before Task 13: move discounting into the baseline only if a plausible positive-rate scenario reverses the model RMSE ranking, changes a model RMSE by at least 5% relative, or changes within-spread share by at least 2 percentage points. Identification diagnostics treat `rho_j` as an unbounded jump-size loading; `|rho_j|>1` is descriptive, not a boundary violation.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / 'notebooks' / '_lib').exists():
    PROJECT_ROOT = ROOT
    NOTEBOOK_DIR = ROOT / 'notebooks'
elif (ROOT / '_lib').exists():
    NOTEBOOK_DIR = ROOT
    PROJECT_ROOT = ROOT.parent
else:
    raise FileNotFoundError('Could not locate notebooks/_lib')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from _lib import chapter3_final, robustness

REMEDIATION_DIR = PROJECT_ROOT / 'excel files' / 'remediation'
OUTPUT_DIR = NOTEBOOK_DIR / 'chapter3_outputs' / 'remediation' / 'robustness'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REGULARISATIONS = (0, 100)


In [ ]:
available = {}
for reg in REGULARISATIONS:
    path = REMEDIATION_DIR / f'calibration_results_reg_{reg}.xlsx'
    if path.exists():
        available[reg] = path
    else:
        print(f'Pending Task 13: {path.name} does not exist yet')
available


In [ ]:
discount_tables = []
promotion_rows = []
identification_tables = []
for reg, path in available.items():
    _, _, svcj_params, test_data = chapter3_final.load_baseline_workbook(path, evaluation_split='test')
    discount = robustness.build_discounting_sensitivity_table(test_data)
    discount['lambda_TS'] = reg
    discount_tables.append(discount)
    decision = robustness.assess_discounting_promotion(discount, scope='All')
    decision['lambda_TS'] = reg
    promotion_rows.append(decision)
    ident = robustness.parameter_identification_summary(svcj_params)
    ident['lambda_TS'] = reg
    identification_tables.append(ident)
    for currency in ('BTC', 'ETH'):
        corr = robustness.parameter_correlation_matrix(svcj_params, currency=currency)
        corr.to_csv(OUTPUT_DIR / f'svcj_parameter_correlations_reg_{reg}_{currency}.csv')

discount_sensitivity = pd.concat(discount_tables, ignore_index=True) if discount_tables else pd.DataFrame()
promotion_decisions = pd.DataFrame(promotion_rows)
identification_summary = pd.concat(identification_tables, ignore_index=True) if identification_tables else pd.DataFrame()
if not discount_sensitivity.empty:
    discount_sensitivity.to_csv(OUTPUT_DIR / 'discounting_sensitivity.csv', index=False)
if not promotion_decisions.empty:
    promotion_decisions.to_csv(OUTPUT_DIR / 'discounting_promotion_decisions.csv', index=False)
if not identification_summary.empty:
    identification_summary.to_csv(OUTPUT_DIR / 'svcj_identification_summary.csv', index=False)
promotion_decisions


## Legacy-versus-corrected identification comparison

The comparison is intentionally conditional on a corrected `lambda_TS=100` workbook existing. Until Task 13 reruns calibration with unbounded `rho_j` and the corrected admissibility rules, the notebook does not manufacture a 'new' parameter path.

In [ ]:
legacy_path = PROJECT_ROOT / 'excel files' / 'calibration_results_reg_100.xlsx'
if 100 in available and legacy_path.exists():
    corrected_svcj = pd.read_excel(available[100], sheet_name='svcj_params')
    legacy_svcj = pd.read_excel(legacy_path, sheet_name='svcj_params')
    legacy_vs_corrected = robustness.compare_identification_summaries(corrected_svcj, legacy_svcj)
    legacy_vs_corrected.to_csv(OUTPUT_DIR / 'svcj_legacy_vs_corrected_identification.csv', index=False)
else:
    legacy_vs_corrected = pd.DataFrame()
legacy_vs_corrected
